In [ ]:
!pip install -q -U google-genai pandas pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not found. ")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

In [ ]:
import os
import re
import json
import time
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm

from google.genai import types

###Main configuration

In [ ]:
# ============================================================
# FILE PATHS
# ============================================================

DATASET_ROOT = Path(
    "/content/drive/MyDrive/processed_dataset"
)

SURVEY_CSV = Path(
    "/content/drive/MyDrive/Summer_training/Gen_AI_test.csv"
)

METADATA_CSV = Path(
    "/content/drive/MyDrive/processed_dataset/"
    "MDT-001_Building_Metadata_Template(Building Metadata).csv"
)

OUTPUT_JSONL_PATH = DATASET_ROOT / "qwen2_vl_albalad_test_dataset.jsonl"

FAILED_JSONL_PATH = DATASET_ROOT / "failed_test_images.jsonl"

RUN_LOG_PATH = DATASET_ROOT / "gemini_test_run.log"


# ============================================================
# MODEL CONFIGURATION
# ============================================================

# Fast and economical model for initial testing.
MODEL_NAME = "gemini-3.5-flash-lite"

# Change to this for a quality comparison:
# MODEL_NAME = "gemini-3.5-flash-lite"


# ============================================================
# TESTING CONFIGURATION
# ============================================================

# Number of images to test from each building.
#
# 5 means:
# - 5 images from Bayt Nassif
# - 5 images from Bayt Noorwali
#
# Set to None later to process every image.
TEST_LIMIT_PER_HOUSE = None

# If False, previously completed images are skipped.
OVERWRITE_EXISTING = False


# ============================================================
# API CONFIGURATION
# ============================================================

MAX_RETRIES = 5
INITIAL_BACKOFF_SECONDS = 10
MAX_BACKOFF_SECONDS = 180

# Adjust this according to your active Gemini API quota.
REQUEST_DELAY_SECONDS = 4.2


# ============================================================
# GENERATION CONFIGURATION
# ============================================================

MAX_OUTPUT_TOKENS = 280
TEMPERATURE = 0.2
TOP_P = 0.9

MIN_CAPTION_WORDS = 20

SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

###Configure only Nassif and Noorwali

In [ ]:
HOUSE_CONFIG = {
    "003_nassif": {
        "path": DATASET_ROOT / "003_nassif",
        "survey_column": "Describe Bayt Nassif",
    },

    "017_noorwali": {
        "path": DATASET_ROOT / "017_noorwali",
        "survey_column": "Describe Bayt Noorwali",
    },
}

###Validate input paths

In [ ]:
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

required_files = {
    "Survey CSV": SURVEY_CSV,
    "Metadata CSV": METADATA_CSV,
}

missing_files = []

for label, file_path in required_files.items():
    if file_path.exists():
        print(f"Found {label}:")
        print(f"  {file_path}")
    else:
        missing_files.append((label, file_path))

for house_id, config in HOUSE_CONFIG.items():
    house_path = Path(config["path"])

    if house_path.exists():
        print(f"Found image folder for {house_id}:")
        print(f"  {house_path}")
    else:
        print(f"Warning: Image folder not found for {house_id}:")
        print(f"  {house_path}")

if missing_files:
    error_message = "\n".join(
        f"{label}: {path}"
        for label, path in missing_files
    )

    raise FileNotFoundError(
        "The following required files were not found:\n"
        f"{error_message}"
    )

print("\nRequired CSV files were found.")

###Load the building metadata CSV

In [ ]:
metadata_df = pd.read_csv(
    METADATA_CSV,
    header=4,
    encoding="utf-8-sig",
)

# Remove completely empty columns such as "Unnamed: 18".
metadata_df = metadata_df.dropna(
    axis=1,
    how="all",
)

# Clean column names.
metadata_df.columns = [
    str(column).strip()
    for column in metadata_df.columns
]

print("Metadata columns:")
print(metadata_df.columns.tolist())

print(f"\nTotal metadata rows: {len(metadata_df)}")

###Select only Nassif and Noorwali metadata

In [ ]:
SELECTED_HOUSE_IDS = list(HOUSE_CONFIG.keys())

selected_metadata_df = metadata_df[
    metadata_df["Building ID"].astype(str).str.strip().isin(
        SELECTED_HOUSE_IDS
    )
].copy()

selected_metadata_df["Building ID"] = (
    selected_metadata_df["Building ID"]
    .astype(str)
    .str.strip()
)

found_ids = set(selected_metadata_df["Building ID"])
missing_ids = set(SELECTED_HOUSE_IDS) - found_ids

if missing_ids:
    raise ValueError(
        "The following building IDs were not found in METADATA_CSV: "
        f"{sorted(missing_ids)}"
    )

display_columns = [
    "Building ID",
    "Building Name (English)",
    "Building Name (Arabic)",
    "District",
    "Street",
    "Building Category",
    "Architectural Style",
    "Construction Material",
    "Number of Floors",
    "Architectural Elements",
]

display(selected_metadata_df[display_columns])

###Clean metadata values

In [ ]:
def clean_text_value(value, default="Not provided"):
    """
    Convert a CSV value into clean plain text.
    """

    if pd.isna(value):
        return default

    text = str(value).strip()

    if not text:
        return default

    if text.lower() in {
        "nan",
        "none",
        "null",
        "n/a",
        "na",
    }:
        return default

    # Replace repeated whitespace and multiline material lists.
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def clean_number_of_floors(value):
    """
    Convert floor values such as 4.0 into '4'.
    """

    if pd.isna(value):
        return "Not provided"

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(int(numeric_value))

        return str(numeric_value)

    except (TypeError, ValueError):
        return clean_text_value(value)

###Create a building metadata lookup

In [ ]:
building_metadata = {}

for _, row in selected_metadata_df.iterrows():
    building_id = clean_text_value(
        row.get("Building ID"),
        default="",
    )

    building_metadata[building_id] = {
        "building_id": building_id,

        "english_name": clean_text_value(
            row.get("Building Name (English)")
        ),

        "arabic_name": clean_text_value(
            row.get("Building Name (Arabic)")
        ),

        "district": clean_text_value(
            row.get("District")
        ),

        "street": clean_text_value(
            row.get("Street")
        ),

        "area": clean_text_value(
            row.get("Area")
        ),

        "city": clean_text_value(
            row.get("City")
        ),

        "gps_coordinates": clean_text_value(
            row.get("GPS Coordinates")
        ),

        "category": clean_text_value(
            row.get("Building Category")
        ),

        "architectural_style": clean_text_value(
            row.get("Architectural Style")
        ),

        "construction_material": clean_text_value(
            row.get("Construction Material")
        ),

        "number_of_floors": clean_number_of_floors(
            row.get("Number of Floors")
        ),

        "architectural_elements": clean_text_value(
            row.get("Architectural Elements")
        ),

        "metadata_lighting": clean_text_value(
            row.get("Lighting Condition")
        ),

        "historical_description": clean_text_value(
            row.get("Historical Description")
        ),

        "reference_source": clean_text_value(
            row.get("Reference Source")
        ),

        "historical_source": clean_text_value(
            row.get("Source of Historical Description")
        ),
    }

print(
    f"Loaded metadata for "
    f"{len(building_metadata)} selected buildings."
)

for building_id, metadata in building_metadata.items():
    print(
        f"- {building_id}: "
        f"{metadata['english_name']} / "
        f"{metadata['arabic_name']}"
    )

###Load human survey descriptions

In [ ]:
survey_df = pd.read_csv(
    SURVEY_CSV,
    encoding="utf-8-sig",
)

print("Survey columns:")

for column in survey_df.columns:
    print(f"- {column}")

In [ ]:
###Clean and combine survey responses

In [ ]:
def clean_survey_response(value):
    """
    Clean one human survey response.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    if text.lower() in {
        "nan",
        "none",
        "null",
        "n/a",
        "na",
    }:
        return None

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def load_survey_descriptions(
    dataframe,
    house_config,
):
    """
    Create one combined survey context for each building.
    """

    survey_lookup = {}

    for house_id, config in house_config.items():
        column_name = config["survey_column"]

        if column_name not in dataframe.columns:
            print(
                f"Warning: Survey column was not found: "
                f"{column_name}"
            )

            survey_lookup[house_id] = (
                "No human survey observations were provided."
            )
            continue

        responses = []

        for value in dataframe[column_name]:
            cleaned_response = clean_survey_response(value)

            if cleaned_response:
                responses.append(cleaned_response)

        # Remove duplicate responses while preserving order.
        unique_responses = list(dict.fromkeys(responses))

        if unique_responses:
            survey_lookup[house_id] = "\n".join(
                f"- {response}"
                for response in unique_responses
            )
        else:
            survey_lookup[house_id] = (
                "No human survey observations were provided."
            )

        print(
            f"{house_id}: "
            f"{len(unique_responses)} unique survey responses"
        )

    return survey_lookup


house_survey_descriptions = load_survey_descriptions(
    survey_df,
    HOUSE_CONFIG,
)

###Preview the combined context

In [ ]:
for house_id in SELECTED_HOUSE_IDS:
    metadata = building_metadata[house_id]

    print("=" * 80)
    print(
        f"{metadata['english_name']} "
        f"({metadata['arabic_name']})"
    )
    print("=" * 80)

    print("\nMetadata:")
    print(
        json.dumps(
            metadata,
            ensure_ascii=False,
            indent=2,
        )
    )

    print("\nSurvey observations:")
    print(house_survey_descriptions[house_id])

    print()

###Prompt template

In [ ]:
PROMPT_TEMPLATE = """
You are an architectural documentation specialist describing a specific
photograph of a historic building in Al-Balad, Jeddah.

BUILDING IDENTIFICATION

Building ID:
{building_id}

Building name in English:
{english_name}

Building name in Arabic:
{arabic_name}

Location:
- District: {district}
- Street: {street}
- Area: {area}
- City: {city}

REFERENCE BUILDING METADATA

Building category:
{category}

Architectural style:
{architectural_style}

Recorded construction materials:
{construction_material}

Recorded number of floors:
{number_of_floors}

Recorded architectural elements:
{architectural_elements}

Historical background:
{historical_description}

HUMAN SURVEY OBSERVATIONS

{human_descriptions}

IMAGE CONTEXT

Folder lighting label:
{lighting}

TASK

Examine this specific image and write one concise architectural description
consisting of exactly 3 or 4 complete sentences.

Rules:

1. The image is the primary source for visible details.

2. The building metadata and human survey responses are supporting context.
Do not state that a contextual feature is visible unless it is supported by
this specific image.

3. Describe visible façade characteristics such as rawashin or roshan
windows, mashrabiya, shish windows, wooden doors, balconies, arches,
decorative plaster, carved details, stonework, masonry, brickwork, or other
architectural elements when they are visible.

4. Describe the visible construction materials cautiously. Do not identify a
material solely from the metadata when the image does not provide enough
visual evidence.

5. Describe the actual viewpoint and composition, such as frontal, oblique,
close-up, wide view, street-level view, upward-looking perspective, partial
façade, or obstructed view.

6. Describe the apparent visual lighting in the photograph. The folder label
is contextual metadata and may be corrected when the image clearly shows a
different condition.

7. Mention surrounding streetscape elements when visible, including pavement,
alleys, trees, planters, benches, barriers, signs, lamps, vehicles, people,
construction equipment, restoration work, and adjacent buildings.

8. Do not invent hidden features, colors, historical events, dates, functions,
materials, floors, objects, or architectural details.

9. Do not include GPS coordinates, reference URLs, source names, or uncertainty
about the prompt.

10. Use professional, objective architectural language.

11. Do not use a title, heading, list, bullet points, Markdown formatting, or
introductory phrase.

12. Return only the final 3-to-4-sentence description.
""".strip()

###Function to build the prompt

In [ ]:
def build_prompt(task):
    """
    Combine metadata, survey observations and image context.
    """

    house_id = task["house_id"]
    metadata = building_metadata[house_id]

    return PROMPT_TEMPLATE.format(
        building_id=metadata["building_id"],
        english_name=metadata["english_name"],
        arabic_name=metadata["arabic_name"],
        district=metadata["district"],
        street=metadata["street"],
        area=metadata["area"],
        city=metadata["city"],
        category=metadata["category"],
        architectural_style=metadata[
            "architectural_style"
        ],
        construction_material=metadata[
            "construction_material"
        ],
        number_of_floors=metadata[
            "number_of_floors"
        ],
        architectural_elements=metadata[
            "architectural_elements"
        ],
        historical_description=metadata[
            "historical_description"
        ],
        human_descriptions=house_survey_descriptions[
            house_id
        ],
        lighting=task["lighting"],
    )

###Discover Day and Night images

In [ ]:
def infer_lighting(image_path, house_directory):
    """
    Infer lighting from parent folder names using specific suffixes.
    """

    relative_parts = [
        part.lower()
        for part in image_path.relative_to(
            house_directory
        ).parts[:-1]
    ]

    for part in relative_parts:
        # Match 'day', 'night' or suffixes like '_d', '_n'
        if "day" in part or part.endswith("_d"):
            return "Day"
        if "night" in part or part.endswith("_n"):
            return "Night"

    return "Unknown"


def create_relative_path(image_path):
    """
    Create a portable path relative to DATASET_ROOT.
    """

    return image_path.relative_to(
        DATASET_ROOT
    ).as_posix()


def discover_house_images(
    house_id,
    config,
):
    """
    Recursively discover supported images for one house.
    """

    house_directory = Path(config["path"])

    if not house_directory.exists():
        print(
            f"Warning: Folder not found for {house_id}: "
            f"{house_directory}"
        )

        return []

    image_files = sorted(
        file_path
        for file_path in house_directory.rglob("*")
        if file_path.is_file()
        and file_path.suffix.lower() in SUPPORTED_EXTENSIONS
    )

    tasks = []

    for image_path in image_files:
        tasks.append(
            {
                "absolute_path": image_path,
                "relative_path": create_relative_path(
                    image_path
                ),
                "house_id": house_id,
                "lighting": infer_lighting(
                    image_path,
                    house_directory,
                ),
            }
        )

    return tasks

###Build a balanced test set

In [ ]:
all_tasks = []
selected_tasks = []

for house_id, config in HOUSE_CONFIG.items():
    house_tasks = discover_house_images(
        house_id,
        config,
    )

    all_tasks.extend(house_tasks)

    if TEST_LIMIT_PER_HOUSE is None:
        limited_tasks = house_tasks
    else:
        limited_tasks = house_tasks[
            :TEST_LIMIT_PER_HOUSE
        ]

    selected_tasks.extend(limited_tasks)

    metadata = building_metadata[house_id]

    print(
        f"{metadata['english_name']}: "
        f"{len(house_tasks)} images found, "
        f"{len(limited_tasks)} selected for this run."
    )

selected_tasks.sort(
    key=lambda task: (
        task["house_id"],
        task["lighting"],
        task["relative_path"],
    )
)

print(f"\nTotal discovered images: {len(all_tasks)}")
print(f"Images selected for testing: {len(selected_tasks)}")

###Show the selected test images

In [ ]:
task_preview = []

for task in selected_tasks:
    metadata = building_metadata[
        task["house_id"]
    ]

    task_preview.append(
        {
            "building_id": task["house_id"],
            "building_name": metadata[
                "english_name"
            ],
            "lighting": task["lighting"],
            "image": task["relative_path"],
        }
    )

task_preview_df = pd.DataFrame(task_preview)

display(task_preview_df)

###Check image sizes

In [ ]:
def inspect_image_sizes(tasks):
    """
    Count dimensions and identify unreadable images.
    """

    size_counts = {}
    invalid_images = []

    for task in tqdm(
        tasks,
        desc="Checking images",
    ):
        try:
            with Image.open(
                task["absolute_path"]
            ) as image:
                size = image.size

            size_counts[size] = (
                size_counts.get(size, 0) + 1
            )

        except Exception as error:
            invalid_images.append(
                {
                    "image": task["relative_path"],
                    "error": str(error),
                }
            )

    return size_counts, invalid_images


size_counts, invalid_images = inspect_image_sizes(
    selected_tasks
)

print("Image dimensions:")

for size, count in sorted(
    size_counts.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(
        f"{size[0]} × {size[1]}: "
        f"{count} images"
    )

if invalid_images:
    print(
        f"\nUnreadable images: "
        f"{len(invalid_images)}"
    )

    display(pd.DataFrame(invalid_images))
else:
    print("\nAll selected images are readable.")

###Load and validate one image

In [ ]:
def load_and_validate_image(image_path):
    """
    Validate and load an image without resizing it.
    """

    # Verify file integrity.
    with Image.open(image_path) as image:
        image.verify()

    # Reopen after verify().
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image)

        if image.mode not in {
            "RGB",
            "RGBA",
        }:
            image = image.convert("RGB")

        image_copy = image.copy()

    return image_copy

###Load completed checkpoints

In [ ]:
def load_processed_images(jsonl_path):
    """
    Read successfully processed image paths from JSONL.
    """

    processed = set()
    malformed_lines = []

    if not jsonl_path.exists():
        return processed, malformed_lines

    with jsonl_path.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

                image_path = record.get("image")

                if image_path:
                    processed.add(image_path)
                else:
                    malformed_lines.append(
                        line_number
                    )

            except json.JSONDecodeError:
                malformed_lines.append(
                    line_number
                )

    return processed, malformed_lines


processed_images, malformed_lines = (
    load_processed_images(
        OUTPUT_JSONL_PATH
    )
)

print(
    f"Previously completed images: "
    f"{len(processed_images)}"
)

if malformed_lines:
    print(
        "Malformed output lines:",
        malformed_lines,
    )

###Select only unfinished images

In [ ]:
if OVERWRITE_EXISTING:
    pending_tasks = selected_tasks.copy()
else:
    pending_tasks = [
        task
        for task in selected_tasks
        if task["relative_path"]
        not in processed_images
    ]

print(
    f"Selected test images: "
    f"{len(selected_tasks)}"
)

print(
    f"Already completed: "
    f"{len(selected_tasks) - len(pending_tasks)}"
)

print(
    f"Remaining images: "
    f"{len(pending_tasks)}"
)

###Clean and validate captions

In [ ]:
def clean_caption(text):
    """
    Remove common unwanted formatting.
    """

    if text is None:
        return ""

    caption = str(text).strip()

    # Remove code fences.
    caption = re.sub(
        r"^```(?:text|markdown)?\s*",
        "",
        caption,
        flags=re.IGNORECASE,
    )

    caption = re.sub(
        r"\s*```$",
        "",
        caption,
    )

    # Remove common headings.
    caption = re.sub(
        r"^(description|architectural description)"
        r"\s*:\s*",
        "",
        caption,
        flags=re.IGNORECASE,
    )

    # Remove an accidental leading bullet.
    caption = re.sub(
        r"^\s*[-*]\s+",
        "",
        caption,
    )

    # Normalize whitespace.
    caption = re.sub(
        r"[ \t]+",
        " ",
        caption,
    )

    caption = re.sub(
        r"\n+",
        " ",
        caption,
    )

    return caption.strip()


def validate_caption(caption):
    """
    Check for clearly incomplete outputs.
    """

    if not caption:
        return False, "Caption is empty."

    word_count = len(caption.split())

    if word_count < MIN_CAPTION_WORDS:
        return (
            False,
            f"Caption contains only "
            f"{word_count} words.",
        )

    return True, None

In [ ]:
def append_jsonl_record(
    file_path,
    record,
):
    """
    Append one JSON object immediately.
    """

    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    serialized = json.dumps(
        record,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    with file_path.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(serialized + "\n")
        file.flush()
        os.fsync(file.fileno())


def append_run_log(message):
    """
    Append a timestamped text log.
    """

    timestamp = datetime.now().isoformat(
        timespec="seconds"
    )

    with RUN_LOG_PATH.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            f"[{timestamp}] {message}\n"
        )


def log_failed_image(
    task,
    error,
    attempts,
):
    """
    Save a permanently failed image.
    """

    failure_record = {
        "image": task["relative_path"],
        "house_id": task["house_id"],
        "lighting": task["lighting"],
        "attempts": attempts,
        "error_type": type(error).__name__,
        "error": str(error),
        "timestamp": datetime.now().isoformat(
            timespec="seconds"
        ),
    }

    append_jsonl_record(
        FAILED_JSONL_PATH,
        failure_record,
    )

In [ ]:
def generate_caption_with_retry(task):
    """
    Generate a caption with exponential retry delays.
    """

    prompt = build_prompt(task)

    image = load_and_validate_image(
        task["absolute_path"]
    )

    last_error = None

    try:
        for attempt in range(
            1,
            MAX_RETRIES + 1,
        ):
            try:
                response = (
                    client.models.generate_content(
                        model=MODEL_NAME,
                        contents=[
                            prompt,
                            image,
                        ],
                        config=types.GenerateContentConfig(
                            temperature=TEMPERATURE,
                            top_p=TOP_P,
                            max_output_tokens=(
                                MAX_OUTPUT_TOKENS
                            ),
                        ),
                    )
                )

                caption = clean_caption(
                    response.text
                )

                is_valid, validation_error = (
                    validate_caption(caption)
                )

                if not is_valid:
                    raise ValueError(
                        validation_error
                    )

                return caption, attempt

            except Exception as error:
                last_error = error

                if attempt >= MAX_RETRIES:
                    break

                exponential_delay = min(
                    INITIAL_BACKOFF_SECONDS
                    * (2 ** (attempt - 1)),
                    MAX_BACKOFF_SECONDS,
                )

                jitter = random.uniform(0, 3)

                wait_seconds = (
                    exponential_delay + jitter
                )

                tqdm.write(
                    f"\nAttempt "
                    f"{attempt}/{MAX_RETRIES} "
                    f"failed for "
                    f"{task['relative_path']}\n"
                    f"{type(error).__name__}: "
                    f"{error}\n"
                    f"Retrying in "
                    f"{wait_seconds:.1f} seconds."
                )

                time.sleep(wait_seconds)

    finally:
        image.close()

    raise RuntimeError(
        f"Caption generation failed after "
        f"{MAX_RETRIES} attempts. "
        f"Last error: {last_error}"
    )

In [ ]:
def create_dataset_record(
    task,
    caption,
):
    """
    Create one Qwen2-VL conversation record.
    """

    metadata = building_metadata[
        task["house_id"]
    ]

    if task["lighting"] == "Unknown":
        user_instruction = (
            "<image>\n"
            "Describe this historic building "
            "in Al-Balad."
        )
    else:
        user_instruction = (
            "<image>\n"
            "Describe this historic building "
            "in Al-Balad during the "
            f"{task['lighting'].lower()}."
        )

    return {
        "image": task["relative_path"],

        "metadata": {
            "building_id": task["house_id"],
            "building_name_english": metadata[
                "english_name"
            ],
            "building_name_arabic": metadata[
                "arabic_name"
            ],
            "lighting": task["lighting"],
            "model": MODEL_NAME,
            "image_width": 224,
            "image_height": 224,
            "caption_source": (
                "Gemini with building metadata "
                "and human survey context"
            ),
        },

        "conversations": [
            {
                "role": "user",
                "content": user_instruction,
            },
            {
                "role": "assistant",
                "content": caption,
            },
        ],
    }

In [ ]:
if not pending_tasks:
    print(
        "No pending test images remain."
    )

else:
    preview_task = pending_tasks[0]

    preview_metadata = building_metadata[
        preview_task["house_id"]
    ]

    print("Selected image:")
    print(preview_task["relative_path"])

    print("\nBuilding:")
    print(
        preview_metadata["english_name"]
    )

    preview_image = load_and_validate_image(
        preview_task["absolute_path"]
    )

    display(preview_image)

    preview_image.close()

    print("\n" + "=" * 80)
    print("COMPLETE PROMPT")
    print("=" * 80)
    print(build_prompt(preview_task))

In [ ]:
if not pending_tasks:
    print(
        "No pending test images remain."
    )

else:
    test_task = pending_tasks[0]

    print(
        f"Generating a caption for:\n"
        f"{test_task['relative_path']}\n"
    )

    test_caption, attempts_used = (
        generate_caption_with_retry(
            test_task
        )
    )

    print(
        f"Attempts used: "
        f"{attempts_used}"
    )

    print("\nGenerated caption:\n")
    print(test_caption)

In [ ]:
def process_dataset(tasks):
    """
    Process all pending tasks sequentially.

    Each successful record is saved immediately.
    """

    successful_count = 0
    failed_count = 0
    total_attempts = 0

    started_at = time.time()

    append_run_log(
        f"Run started; "
        f"model={MODEL_NAME}; "
        f"tasks={len(tasks)}"
    )

    progress_bar = tqdm(
        tasks,
        desc="Generating captions",
        unit="image",
        dynamic_ncols=True,
    )

    for task in progress_bar:
        task_started_at = time.time()

        try:
            caption, attempts_used = (
                generate_caption_with_retry(
                    task
                )
            )

            record = create_dataset_record(
                task,
                caption,
            )

            # Immediate checkpoint.
            append_jsonl_record(
                OUTPUT_JSONL_PATH,
                record,
            )

            successful_count += 1
            total_attempts += attempts_used

            processing_seconds = (
                time.time() - task_started_at
            )

            progress_bar.set_postfix(
                saved=successful_count,
                failed=failed_count,
                seconds=(
                    f"{processing_seconds:.1f}"
                ),
            )

        except Exception as error:
            failed_count += 1

            log_failed_image(
                task=task,
                error=error,
                attempts=MAX_RETRIES,
            )

            tqdm.write(
                f"\nPermanently failed:\n"
                f"{task['relative_path']}\n"
                f"{type(error).__name__}: "
                f"{error}\n"
            )

        # Maintain a minimum request interval.
        elapsed = time.time() - task_started_at

        remaining_delay = (
            REQUEST_DELAY_SECONDS - elapsed
        )

        if remaining_delay > 0:
            time.sleep(remaining_delay)

    total_elapsed = time.time() - started_at

    summary = {
        "successful": successful_count,
        "failed": failed_count,
        "selected_tasks": len(tasks),
        "total_attempts": total_attempts,
        "elapsed_seconds": round(
            total_elapsed,
            2,
        ),
        "average_seconds_per_task": round(
            total_elapsed / max(len(tasks), 1),
            2,
        ),
    }

    append_run_log(
        "Run completed; "
        + json.dumps(
            summary,
            ensure_ascii=False,
        )
    )

    return summary

In [ ]:
if not pending_tasks:
    print(
        "Nothing to process. "
        "All selected test images already exist "
        "in the output JSONL."
    )

else:
    processing_summary = process_dataset(
        pending_tasks
    )

    print("\nProcessing completed.")

    print(
        json.dumps(
            processing_summary,
            ensure_ascii=False,
            indent=2,
        )
    )

    print("\nOutput:")
    print(OUTPUT_JSONL_PATH)

    print("\nFailures:")
    print(FAILED_JSONL_PATH)

In [ ]:
def load_dataset_preview(
    jsonl_path,
    limit=20,
):
    """
    Load records into a readable DataFrame.
    """

    preview_records = []

    if not jsonl_path.exists():
        return pd.DataFrame()

    with jsonl_path.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:
            if not line.strip():
                continue

            record = json.loads(line)

            metadata = record.get(
                "metadata",
                {},
            )

            conversations = record.get(
                "conversations",
                [],
            )

            caption = ""

            if len(conversations) >= 2:
                caption = conversations[1].get(
                    "content",
                    "",
                )

            preview_records.append(
                {
                    "image": record.get("image"),
                    "building_id": metadata.get(
                        "building_id"
                    ),
                    "building_name": metadata.get(
                        "building_name_english"
                    ),
                    "lighting": metadata.get(
                        "lighting"
                    ),
                    "caption": caption,
                }
            )

            if len(preview_records) >= limit:
                break

    return pd.DataFrame(
        preview_records
    )


preview_df = load_dataset_preview(
    OUTPUT_JSONL_PATH,
    limit=20,
)

display(preview_df)

In [ ]:
def validate_dataset_jsonl(jsonl_path):
    """
    Validate syntax, required fields and duplicates.
    """

    valid_records = 0
    invalid_records = []
    duplicate_records = []

    seen_images = set()

    if not jsonl_path.exists():
        print(
            "Output JSONL does not exist."
        )

        return None

    with jsonl_path.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

                image_path = record.get("image")

                if not image_path:
                    raise ValueError(
                        "Missing image path."
                    )

                if image_path in seen_images:
                    duplicate_records.append(
                        {
                            "line": line_number,
                            "image": image_path,
                        }
                    )
                else:
                    seen_images.add(image_path)

                conversations = record.get(
                    "conversations"
                )

                if not isinstance(
                    conversations,
                    list,
                ):
                    raise ValueError(
                        "conversations is not a list."
                    )

                if len(conversations) != 2:
                    raise ValueError(
                        "Expected exactly two "
                        "conversation messages."
                    )

                if (
                    conversations[0].get("role")
                    != "user"
                ):
                    raise ValueError(
                        "First message is not user."
                    )

                if (
                    conversations[1].get("role")
                    != "assistant"
                ):
                    raise ValueError(
                        "Second message is not assistant."
                    )

                caption = conversations[1].get(
                    "content",
                    "",
                ).strip()

                if not caption:
                    raise ValueError(
                        "Assistant caption is empty."
                    )

                valid_records += 1

            except Exception as error:
                invalid_records.append(
                    {
                        "line": line_number,
                        "error": str(error),
                    }
                )

    report = {
        "valid_records": valid_records,
        "invalid_records": len(
            invalid_records
        ),
        "duplicate_records": len(
            duplicate_records
        ),
        "unique_images": len(
            seen_images
        ),
    }

    print(
        json.dumps(
            report,
            indent=2,
        )
    )

    if invalid_records:
        print("\nInvalid records:")
        display(
            pd.DataFrame(
                invalid_records[:20]
            )
        )

    if duplicate_records:
        print("\nDuplicate records:")
        display(
            pd.DataFrame(
                duplicate_records[:20]
            )
        )

    return report


validation_report = validate_dataset_jsonl(
    OUTPUT_JSONL_PATH
)

In [ ]:
final_processed_images, malformed_lines = (
    load_processed_images(
        OUTPUT_JSONL_PATH
    )
)

selected_paths = {
    task["relative_path"]
    for task in selected_tasks
}

completed_selected_paths = (
    selected_paths
    & final_processed_images
)

missing_selected_paths = sorted(
    selected_paths
    - final_processed_images
)

print(
    f"Selected test images: "
    f"{len(selected_paths)}"
)

print(
    f"Completed test images: "
    f"{len(completed_selected_paths)}"
)

print(
    f"Missing test images: "
    f"{len(missing_selected_paths)}"
)

print(
    f"Malformed JSONL lines: "
    f"{len(malformed_lines)}"
)

if missing_selected_paths:
    print("\nMissing images:")

    for image_path in missing_selected_paths:
        print(image_path)
else:
    print(
        "\nAll selected test images "
        "were completed successfully."
    )